# Variant Annotation Pipeline — VEP 115 + ANNOVAR
### Genesis Workbench — Disease Biology Module (v2)

**Extends v1** (ClinVar + ACMG pathogenicity only) with full-spectrum variant annotation:

- **Ensembl VEP 115** — Functional consequence prediction, SIFT/PolyPhen scores, gnomAD allele frequencies, HGVS notation, regulatory annotations
- **ANNOVAR** — Multi-database annotation including gene-based (refGene), population frequencies (gnomAD 4.0), clinical significance (ClinVar), deleteriousness scores (dbNSFP/CADD), and somatic databases (COSMIC)

**Requirements**: Single-node CPU cluster (i3.xlarge or larger). VEP cache and ANNOVAR databases must be pre-installed to UC Volume via `00_setup_reference_databases` notebook.

**Pipeline**:
```
VCF Input → VEP 115 (consequence, SIFT, PolyPhen, gnomAD)
         → ANNOVAR (gene-based, filter-based, CADD)
         → Merge Annotations → Delta Table → MLflow
```

In [0]:
# ── Widget Parameters ──────────────────────────────────────────────────────────
dbutils.widgets.text("catalog", "dhbl_discovery_us_dev", "Catalog")
dbutils.widgets.text("schema", "genesis_schema", "Schema")
dbutils.widgets.text("sql_warehouse_id", "d3fdeafd104a6c26", "SQL Warehouse ID")
dbutils.widgets.text("vcf_path", "", "VCF Input Path")
dbutils.widgets.text("reference_volume", "/Volumes/dhbl_discovery_us_dev/genesis_schema/variant_annotation_reference", "Reference Volume")
dbutils.widgets.dropdown("genome_build", "GRCh38", ["GRCh38", "GRCh37"], "Genome Build")
dbutils.widgets.dropdown("annotation_tools", "both", ["both", "vep_only", "annovar_only"], "Annotation Tools")
dbutils.widgets.text("vep_extra_flags", "", "VEP Extra Flags")
dbutils.widgets.text("annovar_protocols", "refGene,gnomad40_genome,clinvar_20240917,dbnsfp42a,cosmic70", "ANNOVAR Protocols")
dbutils.widgets.text("annovar_operations", "g,f,f,f,f", "ANNOVAR Operations")
dbutils.widgets.text("mlflow_run_id", "", "MLflow Run ID")
dbutils.widgets.text("run_name", "", "Run Name")
dbutils.widgets.text("user_email", "", "User Email")

# ── Read all parameters ──
catalog            = dbutils.widgets.get("catalog")
schema             = dbutils.widgets.get("schema")
sql_warehouse_id   = dbutils.widgets.get("sql_warehouse_id")
vcf_path           = dbutils.widgets.get("vcf_path")
reference_volume   = dbutils.widgets.get("reference_volume")
genome_build       = dbutils.widgets.get("genome_build")
annotation_tools   = dbutils.widgets.get("annotation_tools")
vep_extra_flags    = dbutils.widgets.get("vep_extra_flags")
annovar_protocols  = dbutils.widgets.get("annovar_protocols")
annovar_operations = dbutils.widgets.get("annovar_operations")
mlflow_run_id      = dbutils.widgets.get("mlflow_run_id")
run_name           = dbutils.widgets.get("run_name")
user_email         = dbutils.widgets.get("user_email")

print(f"catalog:            {catalog}")
print(f"schema:             {schema}")
print(f"sql_warehouse_id:   {sql_warehouse_id}")
print(f"vcf_path:           {vcf_path}")
print(f"reference_volume:   {reference_volume}")
print(f"genome_build:       {genome_build}")
print(f"annotation_tools:   {annotation_tools}")
print(f"vep_extra_flags:    {vep_extra_flags}")
print(f"annovar_protocols:  {annovar_protocols}")
print(f"annovar_operations: {annovar_operations}")
print(f"mlflow_run_id:      {mlflow_run_id}")
print(f"run_name:           {run_name}")
print(f"user_email:         {user_email}")

In [0]:
%pip install pysam pyvcf3 pandas --quiet

In [0]:
# ── Read widget values and import libraries ─────────────────────────────────────
catalog            = dbutils.widgets.get("catalog")
schema             = dbutils.widgets.get("schema")
sql_warehouse_id   = dbutils.widgets.get("sql_warehouse_id")
vcf_path           = dbutils.widgets.get("vcf_path")
reference_volume   = dbutils.widgets.get("reference_volume")
genome_build       = dbutils.widgets.get("genome_build")
annotation_tools   = dbutils.widgets.get("annotation_tools")
vep_extra_flags    = dbutils.widgets.get("vep_extra_flags")
annovar_protocols  = dbutils.widgets.get("annovar_protocols")
annovar_operations = dbutils.widgets.get("annovar_operations")
mlflow_run_id      = dbutils.widgets.get("mlflow_run_id")
run_name           = dbutils.widgets.get("run_name")
user_email         = dbutils.widgets.get("user_email")

import os, subprocess, glob, json, shutil, time, re
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import *
import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow.set_tracking_uri("databricks")

# ── Derived paths ──
vep_cache_dir   = f"{reference_volume}/vep_cache"
vep_bin_dir     = f"{reference_volume}/vep_cache/ensembl-vep-release-115"
annovar_dir     = f"{reference_volume}/annovar"
annovar_db_dir  = f"{reference_volume}/annovar/humandb"
reference_fasta = f"{reference_volume}/reference/{genome_build}.fa"
work_dir        = f"/tmp/variant_annotation/{run_name or 'default'}"
os.makedirs(work_dir, exist_ok=True)

# ── Set PERL5LIB for VEP — all module directories required ──
# VEP needs: perl_stubs (DBI/Try::Tiny/Bio::DB::HTS offline stubs),
# its own modules, core Ensembl API, Variation API, FuncGen, IO, and BioPerl.
os.environ["PERL5LIB"] = ":".join([
    os.path.join(vep_bin_dir, "perl_stubs"),              # DBI.pm + offline stubs
    vep_bin_dir,                                           # VEP root
    os.path.join(vep_bin_dir, "modules"),                  # Bio::EnsEMBL::VEP::*
    os.path.join(vep_bin_dir, "ensembl", "modules"),       # Core Ensembl API
    os.path.join(vep_bin_dir, "ensembl-variation", "modules"),  # Variation API
    os.path.join(vep_bin_dir, "ensembl-funcgen", "modules"),    # FuncGen (--regulatory)
    os.path.join(vep_bin_dir, "ensembl-io", "modules"),         # Ensembl IO parsers
    os.path.join(vep_bin_dir, "bioperl", "lib"),                # BioPerl (Bio::PrimarySeqI etc.)
])

# Genome build mapping: GRCh38→hg38 (VEP stays GRCh38), GRCh37→hg19
annovar_buildver = "hg38" if genome_build == "GRCh38" else "hg19"

# Determine which tools to run
run_vep     = True
run_annovar = False  # ANNOVAR disabled — ANNOVAR databases not yet installed
annotation_tools = "vep_only"  # Force VEP-only mode

# Timestamp for this run
run_timestamp = datetime.now(timezone.utc).isoformat()

print(f"Work directory:    {work_dir}")
print(f"VEP cache:         {vep_cache_dir}")
print(f"VEP binary dir:    {vep_bin_dir}")
print(f"ANNOVAR dir:       {annovar_dir}")
print(f"ANNOVAR buildver:  {annovar_buildver}")
print(f"Reference FASTA:   {reference_fasta}")
print(f"PERL5LIB:          {os.environ['PERL5LIB']}")
print(f"Run VEP:           {run_vep}")
print(f"Run ANNOVAR:       {run_annovar}")
print(f"Run timestamp:     {run_timestamp}")

## Validate Prerequisites

In [0]:
# ── Validate all prerequisites before running annotation ────────────────
checks = {}

# VEP cache
vep_species_dir = os.path.join(vep_cache_dir, "homo_sapiens")
checks["VEP cache (homo_sapiens)"] = os.path.isdir(vep_species_dir)

# VEP binary — check both possible locations
vep_binary = os.path.join(vep_bin_dir, "vep")
vep_binary_alt = shutil.which("vep")  # may be on PATH via conda
checks["VEP binary"] = os.path.isfile(vep_binary) or vep_binary_alt is not None

# Resolve actual VEP binary path for later use
if os.path.isfile(vep_binary):
    VEP_BIN = vep_binary
elif vep_binary_alt:
    VEP_BIN = vep_binary_alt
else:
    VEP_BIN = vep_binary  # will fail validation below

# ANNOVAR scripts (disabled — ANNOVAR not yet installed)
# checks["ANNOVAR annotate_variation.pl"] = os.path.isfile(os.path.join(annovar_dir, "annotate_variation.pl"))
# checks["ANNOVAR table_annovar.pl"]      = os.path.isfile(os.path.join(annovar_dir, "table_annovar.pl"))
# checks["ANNOVAR convert2annovar.pl"]    = os.path.isfile(os.path.join(annovar_dir, "convert2annovar.pl"))

# ANNOVAR databases (disabled — ANNOVAR not yet installed)
# checks["ANNOVAR humandb"] = os.path.isdir(annovar_db_dir) and len(os.listdir(annovar_db_dir)) > 0

# VCF input
checks["VCF input file"] = vcf_path.strip() != "" and os.path.isfile(vcf_path)

# Reference FASTA
checks["Reference FASTA"] = os.path.isfile(reference_fasta)

# ── Display results ──
print("=" * 60)
print("  PREREQUISITE VALIDATION")
print("=" * 60)
for check_name, passed in checks.items():
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon}  {check_name}")
print("=" * 60)

# ── Check tool-specific requirements ──
critical_failures = []
if run_vep and not checks["VEP binary"]:
    critical_failures.append("VEP binary not found. Run 00_setup_reference_databases first.")
if run_vep and not checks["VEP cache (homo_sapiens)"]:
    critical_failures.append("VEP cache not found. Run 00_setup_reference_databases first.")
if run_annovar and not checks.get("ANNOVAR table_annovar.pl", False):
    critical_failures.append("ANNOVAR not found. Run 00_setup_reference_databases first.")
if run_annovar and not checks.get("ANNOVAR humandb", False):
    critical_failures.append("ANNOVAR databases not found. Run 00_setup_reference_databases first.")
if not checks["VCF input file"]:
    critical_failures.append(f"VCF input not found at: {vcf_path}")

if critical_failures:
    for msg in critical_failures:
        print(f"\n\u26d4 {msg}")
    raise RuntimeError("Critical prerequisites missing — see above.")

print(f"\n\U0001f7e2 All prerequisites validated. Running: {'VEP + ANNOVAR' if annotation_tools == 'both' else annotation_tools.upper()}")

# ── Start MLflow run ──
experiment_name = f"/Users/{user_email}/genesis-variant-annotation" if user_email else "/Shared/genesis-variant-annotation"
mlflow.set_experiment(experiment_name)

if mlflow_run_id:
    active_run = mlflow.start_run(run_id=mlflow_run_id)
else:
    active_run = mlflow.start_run(run_name=run_name or f"variant_annotation_{genome_build}")

mlflow.log_params({
    "vcf_path": vcf_path,
    "genome_build": genome_build,
    "annotation_tools": annotation_tools,
    "annovar_protocols": annovar_protocols,
    "annovar_operations": annovar_operations,
    "annovar_buildver": annovar_buildver,
    "catalog": catalog,
    "schema": schema,
    "run_timestamp": run_timestamp,
    "pipeline_version": "v2",
})

print(f"MLflow experiment: {experiment_name}")
print(f"MLflow run ID:     {active_run.info.run_id}")

## Stage 1 — Ensembl VEP 115 Annotation

In [0]:
# ── Run Ensembl VEP 115 ────────────────────────────────────────────────────────
vep_df = None
vep_output = os.path.join(work_dir, "vep_output.tsv")
vep_runtime = 0.0

if not run_vep:
    print("\u23ed\ufe0f  VEP skipped (annotation_tools = annovar_only)")
else:
    print("\u25b6 Running Ensembl VEP 115 ...")

    vep_cmd = [
        "perl", VEP_BIN,
        "--input_file", vcf_path,
        "--output_file", vep_output,
        "--format", "vcf",
        "--tab",               # tab-separated output for easy parsing
        "--force_overwrite",
        "--offline",           # use local cache only — no network
        "--cache",
        "--dir_cache", vep_cache_dir,
        "--assembly", genome_build,
        "--species", "homo_sapiens",
        # NOTE: --fasta and --hgvs removed — require htslib (Bio::DB::HTS::Faidx)
        # which is not compiled (setup used --NO_HTSLIB). All other annotations
        # work from cache alone. Re-enable if htslib is compiled in future.
        "--symbol",            # gene symbols (e.g. BRCA1)
        "--biotype",           # transcript biotype
        "--numbers",           # exon/intron numbers
        "--canonical",         # flag canonical transcripts
        "--sift", "b",         # SIFT: both prediction and score
        "--polyphen", "b",     # PolyPhen: both prediction and score
        "--af",                # 1000 Genomes allele freq
        "--af_gnomade",        # gnomAD exomes AF
        "--af_gnomadg",        # gnomAD genomes AF
        "--max_af",            # maximum AF across populations
        "--variant_class",     # variant type (SNV, insertion, etc.)
        "--regulatory",        # regulatory feature consequences
        "--protein",           # protein position
        "--check_existing",    # check known variants (rsIDs)
        "--no_stats",          # skip HTML stats for speed
        "--fork", "4",         # parallel threads
    ]

    # ── CADD plugin (if pre-downloaded to VEP cache) ──
    cadd_snvs = os.path.join(vep_cache_dir, "CADD", "whole_genome_SNVs.tsv.gz")
    if os.path.exists(cadd_snvs):
        cadd_arg = f"CADD,snv={cadd_snvs}"
        cadd_indels = os.path.join(vep_cache_dir, "CADD", "gnomad.genomes.r4.0.indel.tsv.gz")
        if os.path.exists(cadd_indels):
            cadd_arg += f",indels={cadd_indels}"
        vep_cmd.extend(["--plugin", cadd_arg])
        print("  \u2705 CADD plugin enabled")

    # ── User extra flags ──
    if vep_extra_flags.strip():
        vep_cmd.extend(vep_extra_flags.strip().split())

    print(f"  Command: {' '.join(vep_cmd[:8])} ...")
    print(f"  Output:  {vep_output}")

    t0 = time.time()
    proc = subprocess.run(
        vep_cmd,
        capture_output=True,
        text=True,
        timeout=7200,  # 2-hour timeout
    )
    vep_runtime = time.time() - t0

    # Stream last lines of stdout
    if proc.stdout:
        for line in proc.stdout.strip().split("\n")[-10:]:
            print(f"  {line}")

    if proc.returncode != 0:
        print(f"\n\u274c VEP failed (exit code {proc.returncode})")
        if proc.stderr:
            print(proc.stderr[-2000:])
        mlflow.log_metric("vep_exit_code", proc.returncode)
        raise RuntimeError(f"VEP exited with code {proc.returncode}")

    # Count output lines (excluding comment headers)
    with open(vep_output) as f:
        vep_line_count = sum(1 for line in f if not line.startswith("#"))

    print(f"\n\u2705 VEP completed in {vep_runtime:.1f}s — {vep_line_count:,} annotation lines")
    mlflow.log_metrics({
        "vep_variant_count": vep_line_count,
        "vep_runtime_seconds": round(vep_runtime, 1),
        "vep_exit_code": 0,
    })

In [0]:
# ── Parse VEP tab-separated output into DataFrame ────────────────────────────────
if not run_vep:
    print("\u23ed\ufe0f  VEP parsing skipped")
else:
    # Find header line (last line starting with single #, not ##)
    header_line = None
    with open(vep_output) as f:
        for line in f:
            if line.startswith("##"):
                continue
            if line.startswith("#"):
                header_line = line.lstrip("#").strip()
                break

    if header_line is None:
        raise RuntimeError("Could not find VEP header line in output")

    col_names = header_line.split("\t")
    print(f"VEP output columns ({len(col_names)}): {', '.join(col_names[:10])} ...")

    # Parse into DataFrame
    vep_df = pd.read_csv(
        vep_output,
        sep="\t",
        comment="#",
        names=col_names,
        na_values=["-"],
        low_memory=False,
    )

    # ── Build normalized variant key: chr:pos:ref:alt ──
    def make_variant_key_vep(row):
        """Create normalized variant key from VEP output."""
        try:
            uploaded = str(row.get("Uploaded_variation", ""))
            allele = str(row.get("Allele", ""))
            loc = str(row.get("Location", ""))

            # VEP Uploaded_variation format: CHR_POS_REF/ALT
            if "_" in uploaded and uploaded.count("_") >= 2:
                parts = uploaded.split("_")
                chrom = parts[0].replace("chr", "")
                pos = parts[1]
                alleles = "_".join(parts[2:])
                if "/" in alleles:
                    ref, alt = alleles.split("/")[:2]
                else:
                    ref, alt = alleles, allele
                return f"{chrom}:{pos}:{ref}:{alt}"

            # Fallback: Location + Allele
            if ":" in loc:
                chrom, pos_range = loc.split(":", 1)
                chrom = chrom.replace("chr", "")
                pos = pos_range.split("-")[0]
                return f"{chrom}:{pos}:.:{allele}"

            return None
        except Exception:
            return None

    vep_df["variant_key"] = vep_df.apply(make_variant_key_vep, axis=1)

    # ── Summary ──
    print(f"\nVEP annotations parsed: {len(vep_df):,} rows")
    print(f"Unique variant keys:    {vep_df['variant_key'].nunique():,}")

    if "Consequence" in vep_df.columns:
        print(f"\nConsequence distribution (top 10):")
        all_consequences = vep_df["Consequence"].dropna().str.split(",").explode()
        display(all_consequences.value_counts().head(10).to_frame("count"))

    if "IMPACT" in vep_df.columns:
        print(f"\nImpact distribution:")
        display(vep_df["IMPACT"].value_counts().to_frame("count"))

## Stage 2 — ANNOVAR Annotation

In [0]:
# ── Run ANNOVAR: VCF → convert2annovar → table_annovar ──────────────────────────────
annovar_df = None
annovar_runtime = 0.0

if not run_annovar:
    print("\u23ed\ufe0f  ANNOVAR skipped (annotation_tools = vep_only)")
else:
    print("\u25b6 Running ANNOVAR ...")

    # Step 1: Convert VCF to ANNOVAR input format
    avinput_path = os.path.join(work_dir, "annovar_input.avinput")
    convert_cmd = [
        "perl", os.path.join(annovar_dir, "convert2annovar.pl"),
        "-format", "vcf4",
        vcf_path,
        "-outfile", avinput_path,
        "-includeinfo",
        "-withzyg",
    ]

    print(f"  Converting VCF \u2192 ANNOVAR format ...")
    proc_convert = subprocess.run(
        convert_cmd, capture_output=True, text=True, timeout=1800
    )

    if proc_convert.returncode != 0:
        print(f"\u274c convert2annovar.pl failed (exit {proc_convert.returncode})")
        if proc_convert.stderr:
            print(proc_convert.stderr[-1000:])
        raise RuntimeError("ANNOVAR VCF conversion failed")

    avinput_lines = sum(1 for _ in open(avinput_path))
    print(f"  \u2705 Converted {avinput_lines:,} variants to ANNOVAR format")

    # Step 2: Run table_annovar.pl
    annovar_output_prefix = os.path.join(work_dir, "annovar_output")
    annovar_cmd = [
        "perl", os.path.join(annovar_dir, "table_annovar.pl"),
        avinput_path,
        annovar_db_dir,
        "-buildver", annovar_buildver,
        "-out", annovar_output_prefix,
        "-protocol", annovar_protocols,
        "-operation", annovar_operations,
        "-nastring", ".",
        "-remove",  # clean up temp files
    ]

    print(f"  Protocols:  {annovar_protocols}")
    print(f"  Operations: {annovar_operations}")
    print(f"  Build:      {annovar_buildver}")

    t0 = time.time()
    proc_annovar = subprocess.run(
        annovar_cmd, capture_output=True, text=True, timeout=7200
    )
    annovar_runtime = time.time() - t0

    # Stream last lines of stdout
    if proc_annovar.stdout:
        for line in proc_annovar.stdout.strip().split("\n")[-10:]:
            print(f"  {line}")

    if proc_annovar.returncode != 0:
        print(f"\n\u274c ANNOVAR failed (exit code {proc_annovar.returncode})")
        if proc_annovar.stderr:
            print(proc_annovar.stderr[-2000:])
        mlflow.log_metric("annovar_exit_code", proc_annovar.returncode)
        raise RuntimeError(f"ANNOVAR exited with code {proc_annovar.returncode}")

    print(f"\n\u2705 ANNOVAR completed in {annovar_runtime:.1f}s")
    mlflow.log_metrics({
        "annovar_runtime_seconds": round(annovar_runtime, 1),
        "annovar_exit_code": 0,
    })

In [0]:
# ── Parse ANNOVAR multianno output into DataFrame ─────────────────────────────
if not run_annovar:
    print("\u23ed\ufe0f  ANNOVAR parsing skipped")
else:
    # Find the multianno output file
    multianno_pattern = os.path.join(work_dir, f"annovar_output.{annovar_buildver}_multianno.txt")
    if not os.path.exists(multianno_pattern):
        # Try glob as fallback
        multianno_files = glob.glob(os.path.join(work_dir, "annovar_output*multianno*"))
        if not multianno_files:
            raise RuntimeError(
                f"ANNOVAR multianno output not found. Files in work_dir: {os.listdir(work_dir)}"
            )
        multianno_pattern = multianno_files[0]

    print(f"Reading: {os.path.basename(multianno_pattern)}")

    annovar_df = pd.read_csv(
        multianno_pattern, sep="\t", low_memory=False, na_values="."
    )

    # ── Build normalized variant key: chr:pos:ref:alt ──
    def make_variant_key_annovar(row):
        """Create normalized variant key from ANNOVAR output."""
        try:
            chrom = str(row["Chr"]).replace("chr", "")
            pos = str(int(row["Start"]))
            ref = str(row["Ref"])
            alt = str(row["Alt"])
            return f"{chrom}:{pos}:{ref}:{alt}"
        except Exception:
            return None

    annovar_df["variant_key"] = annovar_df.apply(make_variant_key_annovar, axis=1)

    annovar_variant_count = len(annovar_df)
    mlflow.log_metric("annovar_variant_count", annovar_variant_count)

    print(f"ANNOVAR annotations parsed: {annovar_variant_count:,} variants")
    print(f"Columns: {list(annovar_df.columns)}")

    if "Func.refGene" in annovar_df.columns:
        print(f"\nFunctional distribution:")
        display(annovar_df["Func.refGene"].value_counts().head(10).to_frame("count"))

    if "ExonicFunc.refGene" in annovar_df.columns:
        exonic = annovar_df["ExonicFunc.refGene"].dropna()
        if len(exonic) > 0:
            print(f"\nExonic function distribution:")
            display(exonic.value_counts().head(10).to_frame("count"))

## Stage 3 — Merge Annotations

In [0]:
# ── Merge VEP + ANNOVAR annotations ─────────────────────────────────────────
print("\u25b6 Merging annotations ...")

if annotation_tools == "vep_only":
    merged_df = vep_df.copy()
    merged_df["annotation_source"] = "vep"
    print(f"  VEP-only mode: {len(merged_df):,} rows")

elif annotation_tools == "annovar_only":
    merged_df = annovar_df.copy()
    merged_df["annotation_source"] = "annovar"
    print(f"  ANNOVAR-only mode: {len(merged_df):,} rows")

else:
    # Both tools — merge on variant_key
    # VEP may have multiple rows per variant (one per transcript)
    # ANNOVAR has one row per variant
    # Strategy: keep VEP granularity (transcript-level), left-join ANNOVAR

    # Prefix ANNOVAR columns to avoid name collisions
    skip_cols = {"variant_key", "Chr", "Start", "End", "Ref", "Alt"}
    annovar_cols_to_merge = [c for c in annovar_df.columns if c not in skip_cols]
    annovar_for_merge = annovar_df[["variant_key"] + annovar_cols_to_merge].copy()
    annovar_for_merge.columns = ["variant_key"] + [
        f"annovar_{c}" for c in annovar_cols_to_merge
    ]

    # Deduplicate ANNOVAR rows per variant key (keep first)
    annovar_for_merge = annovar_for_merge.drop_duplicates(
        subset="variant_key", keep="first"
    )

    merged_df = vep_df.merge(
        annovar_for_merge, on="variant_key", how="outer", indicator=True
    )

    # Summarize merge
    merge_stats = merged_df["_merge"].value_counts()
    both_count = merge_stats.get("both", 0)
    vep_only_count = merge_stats.get("left_only", 0)
    annovar_only_count = merge_stats.get("right_only", 0)

    merged_df["annotation_source"] = merged_df["_merge"].map({
        "both": "vep+annovar",
        "left_only": "vep_only",
        "right_only": "annovar_only",
    })
    merged_df.drop(columns=["_merge"], inplace=True)

    print(f"  Matched (both):   {both_count:,}")
    print(f"  VEP only:         {vep_only_count:,}")
    print(f"  ANNOVAR only:     {annovar_only_count:,}")
    print(f"  Total merged:     {len(merged_df):,}")

# ── Combined impact column ───────────────────────────────────────────────────
# Prioritize VEP IMPACT; fall back to ANNOVAR ExonicFunc mapping
def compute_combined_impact(row):
    """Derive impact level: VEP IMPACT > ANNOVAR ExonicFunc > MODIFIER."""
    vep_impact = str(row.get("IMPACT", "")).strip()
    if vep_impact in ("HIGH", "MODERATE", "LOW", "MODIFIER"):
        return vep_impact

    # ANNOVAR ExonicFunc heuristic mapping
    exonic = str(
        row.get("annovar_ExonicFunc.refGene", row.get("ExonicFunc.refGene", ""))
    ).lower()
    if any(x in exonic for x in ["frameshift", "stopgain", "stoploss", "splicing"]):
        return "HIGH"
    elif any(x in exonic for x in ["nonsynonymous", "nonframeshift"]):
        return "MODERATE"
    elif "synonymous" in exonic and "non" not in exonic:
        return "LOW"
    return "MODIFIER"

merged_df["combined_impact"] = merged_df.apply(compute_combined_impact, axis=1)

# ── Add run metadata columns ──
merged_df["run_name"] = run_name
merged_df["annotated_at"] = run_timestamp
merged_df["genome_build"] = genome_build
merged_df["tools_used"] = annotation_tools

print(f"\n\u2705 Merge complete: {len(merged_df):,} total annotation rows")
print(f"\nCombined impact distribution:")
display(merged_df["combined_impact"].value_counts().to_frame("count"))

## Stage 4 — Save to Delta and MLflow

In [0]:
# ── Save to Delta tables with merge-by-run_name (idempotent) ─────────────────
full_table_prefix = f"{catalog}.{schema}"

def save_with_merge(pdf, table_name):
    """Save pandas DataFrame to Delta using delete-then-append for run_name idempotency."""
    full_table = f"{full_table_prefix}.{table_name}"

    # Convert all columns to string to avoid Arrow serialization issues with mixed types
    pdf_clean = pdf.copy()
    for col in pdf_clean.columns:
        pdf_clean[col] = pdf_clean[col].astype(str).replace("nan", None).replace("None", None)

    sdf = spark.createDataFrame(pdf_clean)

    # Check if table exists
    try:
        spark.sql(f"DESCRIBE TABLE {full_table}")
        table_exists = True
    except Exception:
        table_exists = False

    if not table_exists:
        sdf.write.format("delta").mode("overwrite").saveAsTable(full_table)
        count = spark.table(full_table).count()
        print(f"  \u2705 Created {full_table} \u2014 {count:,} rows")
    else:
        # Delete existing rows for this run_name, then append (idempotent)
        spark.sql(f"DELETE FROM {full_table} WHERE run_name = '{run_name}'")
        sdf.write.format("delta").mode("append").saveAsTable(full_table)
        count = spark.table(full_table).filter(F.col("run_name") == run_name).count()
        print(f"  \u2705 Merged into {full_table} \u2014 {count:,} rows for run '{run_name}'")

    return count

print("\u25b6 Saving results to Delta tables ...\n")

# ── Table 1: VEP annotations ──
vep_row_count = 0
if vep_df is not None:
    vep_save = vep_df.copy()
    vep_save["run_name"] = run_name
    vep_save["annotated_at"] = run_timestamp
    vep_save["genome_build"] = genome_build
    vep_row_count = save_with_merge(vep_save, "vep_annotations")
else:
    print("  \u23ed\ufe0f  vep_annotations \u2014 skipped (VEP not run)")

# ── Table 2: ANNOVAR annotations ──
annovar_row_count = 0
if annovar_df is not None:
    annovar_save = annovar_df.copy()
    annovar_save["run_name"] = run_name
    annovar_save["annotated_at"] = run_timestamp
    annovar_save["genome_build"] = genome_build
    annovar_row_count = save_with_merge(annovar_save, "annovar_annotations")
else:
    print("  \u23ed\ufe0f  annovar_annotations \u2014 skipped (ANNOVAR not run)")

# ── Table 3: Unified merged annotations ──
merged_row_count = save_with_merge(merged_df, "variant_annotations_v2")

print(f"\n{'=' * 50}")
print(f"  Delta Save Summary")
print(f"{'=' * 50}")
print(f"  VEP annotations:      {vep_row_count:,} rows")
print(f"  ANNOVAR annotations:  {annovar_row_count:,} rows")
print(f"  Merged (v2):          {merged_row_count:,} rows")
print(f"{'=' * 50}")

In [0]:
# ── MLflow summary logging ─────────────────────────────────────────────────────
total_variants = (
    merged_df["variant_key"].nunique()
    if "variant_key" in merged_df.columns
    else len(merged_df)
)

mlflow.log_metrics({
    "total_unique_variants": total_variants,
    "total_annotation_rows": len(merged_df),
    "vep_rows": len(vep_df) if vep_df is not None else 0,
    "annovar_rows": len(annovar_df) if annovar_df is not None else 0,
    "merged_rows": len(merged_df),
    "total_runtime_seconds": round(vep_runtime + annovar_runtime, 1),
})

# ── Log artifacts ──
# Consequence distribution
if vep_df is not None and "Consequence" in vep_df.columns:
    consq_dist = (
        vep_df["Consequence"].dropna().str.split(",").explode().value_counts().to_dict()
    )
    consq_path = os.path.join(work_dir, "consequence_distribution.json")
    with open(consq_path, "w") as f:
        json.dump(consq_dist, f, indent=2)
    mlflow.log_artifact(consq_path)

# Impact distribution
impact_dist = merged_df["combined_impact"].value_counts().to_dict()
impact_path = os.path.join(work_dir, "impact_distribution.json")
with open(impact_path, "w") as f:
    json.dump(impact_dist, f, indent=2)
mlflow.log_artifact(impact_path)

# Top pathogenic variants (HIGH impact)
high_impact = merged_df[merged_df["combined_impact"] == "HIGH"]
if len(high_impact) > 0:
    top_path = os.path.join(work_dir, "top_pathogenic_variants.csv")
    high_impact.head(500).to_csv(top_path, index=False)
    mlflow.log_artifact(top_path)

# Set completion tags
mlflow.set_tags({
    "annotation_tools": annotation_tools,
    "genome_build": genome_build,
    "annotation_complete": "true",
    "pipeline_version": "v2",
    "user_email": user_email or "unknown",
})

mlflow.end_run()

# ── Final summary box ──
run_id = active_run.info.run_id
w = 62
print()
print("\u2554" + "\u2550" * w + "\u2557")
print("\u2551" + " VARIANT ANNOTATION COMPLETE".center(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")
print("\u2551" + f"  Run name:         {run_name}".ljust(w) + "\u2551")
print("\u2551" + f"  MLflow run:       {run_id[:24]}...".ljust(w) + "\u2551")
print("\u2551" + f"  Genome build:     {genome_build}".ljust(w) + "\u2551")
print("\u2551" + f"  Tools:            {annotation_tools}".ljust(w) + "\u2551")
print("\u2551" + f"  VEP variants:     {len(vep_df) if vep_df is not None else 0:,}".ljust(w) + "\u2551")
print("\u2551" + f"  ANNOVAR variants:  {len(annovar_df) if annovar_df is not None else 0:,}".ljust(w) + "\u2551")
print("\u2551" + f"  Merged rows:      {len(merged_df):,}".ljust(w) + "\u2551")
print("\u2551" + f"  HIGH impact:      {len(high_impact):,}".ljust(w) + "\u2551")
print("\u2551" + f"  Total runtime:    {vep_runtime + annovar_runtime:.1f}s".ljust(w) + "\u2551")
print("\u2560" + "\u2550" * w + "\u2563")
print("\u2551" + f"  Delta tables:".ljust(w) + "\u2551")
print("\u2551" + f"    {catalog}.{schema}.vep_annotations".ljust(w) + "\u2551")
print("\u2551" + f"    {catalog}.{schema}.annovar_annotations".ljust(w) + "\u2551")
print("\u2551" + f"    {catalog}.{schema}.variant_annotations_v2".ljust(w) + "\u2551")
print("\u255a" + "\u2550" * w + "\u255d")

## Configuration Reference

### ANNOVAR Protocol Options

| Protocol | Type | Description |
| --- | --- | --- |
| `refGene` | Gene-based (g) | NCBI RefSeq gene annotations |
| `ensGene` | Gene-based (g) | Ensembl gene annotations |
| `knownGene` | Gene-based (g) | UCSC Known Genes |
| `gnomad40_genome` | Filter-based (f) | gnomAD v4.0 genome allele frequencies |
| `gnomad40_exome` | Filter-based (f) | gnomAD v4.0 exome allele frequencies |
| `clinvar_20240917` | Filter-based (f) | ClinVar clinical significance (Sept 2024) |
| `dbnsfp42a` | Filter-based (f) | dbNSFP v4.2a \~ SIFT, PolyPhen2, CADD, REVEL, etc. |
| `cosmic70` | Filter-based (f) | COSMIC somatic mutations |
| `avsnp150` | Filter-based (f) | dbSNP 150 rsIDs |
| `cadd13` | Filter-based (f) | CADD v1.3 precomputed scores |

### VEP 115 Plugin Options

| Plugin | Description |
| --- | --- |
| `CADD` | Combined Annotation Dependent Depletion scores |
| `REVEL` | Rare Exome Variant Ensemble Learner |
| `SpliceAI` | Deep learning splice variant predictions |
| `dbNSFP` | Database of Non-synonymous Functional Predictions |
| `AlphaMissense` | DeepMind protein structure\~based pathogenicity |
| `LoF` | Loss-of-function transcript effect predictor (LOFTEE) |

### Links

* [Ensembl VEP 115 Documentation](https://www.ensembl.org/info/docs/tools/vep/index.html)
* [ANNOVAR Documentation](https://annovar.openbioinformatics.org/en/latest/)
* [VEP Plugins](https://www.ensembl.org/info/docs/tools/vep/script/vep_plugins.html)
* [ANNOVAR Database Download](https://annovar.openbioinformatics.org/en/latest/user-guide/download/)

> **Licensing**: ANNOVAR requires registration and is licensed for academic/non-commercial use. Contact the [ANNOVAR team](https://annovar.openbioinformatics.org/en/latest/user-guide/download/) for commercial licensing.